<a href="https://colab.research.google.com/github/ysuter/FHNW-BAI-DataWrangling/blob/main/session6_apis_uebungen.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Session 6 – Web-APIs: Praktische Übungen
**Data Engineering and Wrangling | BSc Business AI | FHNW FS 2026**

---

In dieser Übungsserie arbeitest du mit drei APIs unterschiedlicher Komplexität:

| # | API | Authentifizierung | Schwierigkeitsgrad |
|---|-----|-------------------|--------------------|
| 1 | Fake Store API | Kein Key nötig | ⭐ Einstieg |
| 2 | Open-Meteo API | Kein Key nötig | ⭐⭐ Mittelstufe |
| 3 | GitHub REST API | Optional (Token empfohlen) | ⭐⭐⭐ Fortgeschritten |

**Hinweis:** Musterlösungen sind in ausgeblendeten Zellen (mit `# MUSTERLÖSUNG` markiert) vorhanden.

In [ ]:
# Benötigte Libraries installieren (falls noch nicht vorhanden)
# !pip install requests pandas matplotlib python-dotenv

import requests
import pandas as pd
import matplotlib.pyplot as plt
import json
import time
import os

print("Libraries geladen ✓")

---
## 📦 Teil 1: Fake Store API
**URL:** https://fakestoreapi.com  
**Dokumentation:** https://fakestoreapi.com/docs

Die Fake Store API simuliert einen Online-Shop und bietet Endpunkte für Produkte, Warenkorb und Benutzer.
Sie benötigt keinen API Key – ideal zum Einstieg.

### Nützliche Endpunkte
- `GET /products` – Alle Produkte
- `GET /products/{id}` – Einzelnes Produkt
- `GET /products?limit=5` – Nur 5 Produkte
- `GET /products/categories` – Alle Kategorien
- `GET /products/category/{category}` – Produkte einer Kategorie

### Aufgabe 1.1 – Alle Produkte abrufen
Rufe alle Produkte von `https://fakestoreapi.com/products` ab.
- Prüfe den Status Code
- Gib die Anzahl der Produkte aus
- Zeige den Titel und Preis des ersten Produkts an

In [ ]:
# Deine lösung hier:



In [ ]:
# MUSTERLÖSUNG 1.1
BASE_URL = "https://fakestoreapi.com"

response = requests.get(f"{BASE_URL}/products")

print(f"Status Code: {response.status_code}")

products = response.json()
print(f"Anzahl Produkte: {len(products)}")

erstes = products[0]
print(f"\nErstes Produkt:")
print(f"  Titel: {erstes['title']}")
print(f"  Preis: ${erstes['price']}")
print(f"  Kategorie: {erstes['category']}")

### Aufgabe 1.2 – DataFrame erstellen
Wandle die Produktliste in einen pandas DataFrame um.
- Zeige nur die Spalten: `id`, `title`, `price`, `category`, `rating`
- Das `rating`-Feld ist ein verschachteltes Dict – extrahiere nur den Wert `rate`
- Sortiere nach Preis (absteigend) und zeige die Top-5

In [ ]:
# Deine Lösung hier:
# Hinweis: Du kannst products aus Aufgabe 1.1 wiederverwenden



In [ ]:
# MUSTERLÖSUNG 1.2
df = pd.DataFrame(products)

# Rating-Wert aus verschachteltem Dict extrahieren
df['rating_score'] = df['rating'].apply(lambda x: x['rate'])
df['rating_count'] = df['rating'].apply(lambda x: x['count'])

# Nur relevante Spalten
df_clean = df[['id', 'title', 'price', 'category', 'rating_score', 'rating_count']]

# Top 5 nach Preis
print("Top 5 teuerste Produkte:")
print(df_clean.sort_values('price', ascending=False).head())

### Aufgabe 1.3 – Filtern nach Kategorie
- Lade alle verfügbaren Kategorien (`GET /products/categories`)
- Wähle die Kategorie `"electronics"` und lade nur diese Produkte
- Finde das günstigste Elektronik-Produkt

In [ ]:
# Deine Lösung hier:



In [ ]:
# MUSTERLÖSUNG 1.3
# Kategorien laden
cats_resp = requests.get(f"{BASE_URL}/products/categories")
kategorien = cats_resp.json()
print(f"Verfügbare Kategorien: {kategorien}")

# Elektronik-Produkte laden
elec_resp = requests.get(f"{BASE_URL}/products/category/electronics")
elektronik = elec_resp.json()
print(f"\nAnzahl Elektronik-Produkte: {len(elektronik)}")

# Günstigstes Produkt
df_elec = pd.DataFrame(elektronik)
guenstigstes = df_elec.loc[df_elec['price'].idxmin()]
print(f"\nGünstigstes Elektronik-Produkt:")
print(f"  {guenstigstes['title']}")
print(f"  Preis: ${guenstigstes['price']}")

### Aufgabe 1.4 – Query Parameter verwenden
Rufe nur **3 Produkte** ab, **sortiert absteigend** (`sort=desc`).
Verwende dazu den `params`-Parameter von `requests.get()` – **nicht** direkt die URL.

In [ ]:
# Deine Lösung hier:



In [ ]:
# MUSTERLÖSUNG 1.4
params = {
    "limit": 3,
    "sort": "desc"
}

resp = requests.get(f"{BASE_URL}/products", params=params)
print(f"Anfrage-URL: {resp.url}")  # URL mit kodierten Parametern anzeigen

drei_produkte = resp.json()
for p in drei_produkte:
    print(f"  ID {p['id']}: {p['title'][:50]}...")

---
## 🌤️ Teil 2: Open-Meteo API
**URL:** https://api.open-meteo.com  
**Dokumentation:** https://open-meteo.com/en/docs

Open-Meteo ist eine kostenlose Wetter-API ohne Registrierung. Sie liefert stündliche und tägliche Wetterdaten weltweit.

### Koordinaten für unsere Städte
| Stadt | Latitude | Longitude |
|-------|----------|----------|
| Olten | 47.352 | 7.903 |
| Zürich | 47.377 | 8.540 |
| Basel | 47.560 | 7.590 |

### Wichtige Parameter
- `latitude`, `longitude` – Koordinaten (Pflicht)
- `hourly` – Stündliche Variablen (z.B. `temperature_2m`)
- `daily` – Tägliche Variablen (z.B. `temperature_2m_max`, `precipitation_sum`)
- `timezone` – z.B. `Europe/Zurich`
- `forecast_days` – Anzahl Vorhersagetage (1–16)

### Aufgabe 2.1 – Aktuelle Wetterdaten für Olten
Rufe die **stündliche** Temperatur (`temperature_2m`) und den Niederschlag (`precipitation`) für Olten ab.
- Verwende `forecast_days=1`
- Zeige die Struktur der Response
- Erstelle einen DataFrame mit Zeit und Temperatur

In [ ]:
# Deine Lösung hier:



In [ ]:
# MUSTERLÖSUNG 2.1
METEO_URL = "https://api.open-meteo.com/v1/forecast"

params = {
    "latitude": 47.352,
    "longitude": 7.903,
    "hourly": "temperature_2m,precipitation",
    "timezone": "Europe/Zurich",
    "forecast_days": 1
}

resp = requests.get(METEO_URL, params=params)
resp.raise_for_status()
data = resp.json()

print("Keys in der Response:", list(data.keys()))
print("Hourly Keys:", list(data['hourly'].keys()))

# DataFrame erstellen
df_wetter = pd.DataFrame({
    'Zeit': pd.to_datetime(data['hourly']['time']),
    'Temperatur_C': data['hourly']['temperature_2m'],
    'Niederschlag_mm': data['hourly']['precipitation']
})

print(f"\nWetterdaten für Olten (heute):")
print(df_wetter.head(6))
print(f"\nAktuelle Temperatur: {df_wetter.iloc[0]['Temperatur_C']}°C")

### Aufgabe 2.2 – 7-Tage-Prognose
Lade die **tägliche** 7-Tage-Prognose für Olten mit folgenden Variablen:
- `temperature_2m_max` und `temperature_2m_min`
- `precipitation_sum`
- `weathercode`

Erstelle einen DataFrame und berechne:
- Den wärmsten und kältesten Tag
- Den gesamten Niederschlag der Woche

In [ ]:
# Deine Lösung hier:



In [ ]:
# MUSTERLÖSUNG 2.2
params_7d = {
    "latitude": 47.352,
    "longitude": 7.903,
    "daily": "temperature_2m_max,temperature_2m_min,precipitation_sum,weathercode",
    "timezone": "Europe/Zurich",
    "forecast_days": 7
}

resp7 = requests.get(METEO_URL, params=params_7d)
resp7.raise_for_status()
data7 = resp7.json()

df_7d = pd.DataFrame({
    'Datum': pd.to_datetime(data7['daily']['time']),
    'T_max': data7['daily']['temperature_2m_max'],
    'T_min': data7['daily']['temperature_2m_min'],
    'Niederschlag_mm': data7['daily']['precipitation_sum'],
    'Wettercode': data7['daily']['weathercode']
})

print("7-Tage-Prognose Olten:")
print(df_7d.to_string(index=False))

print(f"\nWärmster Tag: {df_7d.loc[df_7d['T_max'].idxmax(), 'Datum'].date()} ({df_7d['T_max'].max()}°C)")
print(f"Kältester Tag: {df_7d.loc[df_7d['T_min'].idxmin(), 'Datum'].date()} ({df_7d['T_min'].min()}°C)")
print(f"Gesamtniederschlag: {df_7d['Niederschlag_mm'].sum():.1f} mm")

### Aufgabe 2.3 – Visualisierung
Erstelle einen **Linienplot** der Temperatur-Prognose:
- X-Achse: Datum
- Y-Achse: Temperatur in °C
- Zwei Linien: Max- und Min-Temperatur
- Fülle den Bereich zwischen den Linien (Tipp: `plt.fill_between`)

In [ ]:
# Deine Lösung hier:



In [ ]:
# MUSTERLÖSUNG 2.3
fig, ax = plt.subplots(figsize=(10, 5))

ax.plot(df_7d['Datum'], df_7d['T_max'], color='#E74C3C', marker='o', label='Max-Temperatur')
ax.plot(df_7d['Datum'], df_7d['T_min'], color='#3498DB', marker='o', label='Min-Temperatur')
ax.fill_between(df_7d['Datum'], df_7d['T_min'], df_7d['T_max'],
                alpha=0.15, color='#9B59B6', label='Temperaturbereich')

ax.set_title('7-Tage-Temperaturprognose Olten', fontsize=14, fontweight='bold')
ax.set_xlabel('Datum')
ax.set_ylabel('Temperatur (°C)')
ax.legend()
ax.grid(True, alpha=0.3)
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()

---
## 🐙 Teil 3: GitHub REST API
**URL:** https://api.github.com  
**Dokumentation:** https://docs.github.com/en/rest

Die GitHub API erlaubt Zugriff auf Repositories, Issues, User-Daten und mehr.

### Rate Limits
| Modus | Requests/Stunde |
|-------|----------------|
| Ohne Token | 60 |
| Mit Token | 5000 |

### Optional: GitHub Token erstellen
1. GitHub.com → Settings → Developer Settings → Personal Access Tokens
2. Token (classic) → Generate → Scope: `public_repo` genügt
3. In `.env` Datei speichern: `GITHUB_TOKEN=ghp_...`

In [ ]:
# Optional: Token aus .env laden
# from dotenv import load_dotenv
# load_dotenv()
# GITHUB_TOKEN = os.environ.get("GITHUB_TOKEN")

# Für dieses Notebook arbeiten wir ohne Token
# (60 Requests/Stunde genügen für die Übungen)
GITHUB_TOKEN = None  # Hier eigenen Token eintragen oder None lassen

GITHUB_BASE = "https://api.github.com"

# Headers vorbereiten
headers = {"Accept": "application/vnd.github.v3+json"}
if GITHUB_TOKEN:
    headers["Authorization"] = f"Bearer {GITHUB_TOKEN}"
    print("✓ Mit Token authentifiziert (5000 Req/h)")
else:
    print("⚠ Ohne Token (60 Req/h) – für die Übungen ausreichend")

### Aufgabe 3.1 – Rate Limit prüfen
Rufe `GET /rate_limit` ab und zeige an:
- Wie viele Requests sind noch verfügbar?
- Wann wird das Limit zurückgesetzt?

In [ ]:
# Deine Lösung hier:



In [ ]:
# MUSTERLÖSUNG 3.1
import datetime

resp_limit = requests.get(f"{GITHUB_BASE}/rate_limit", headers=headers)
resp_limit.raise_for_status()
limit_data = resp_limit.json()

core = limit_data['resources']['core']
reset_time = datetime.datetime.fromtimestamp(core['reset'])

print(f"Rate Limit:")
print(f"  Limit:     {core['limit']} Requests/Stunde")
print(f"  Verbraucht: {core['limit'] - core['remaining']}")
print(f"  Verbleibend: {core['remaining']}")
print(f"  Reset um:  {reset_time.strftime('%H:%M:%S')} Uhr")

### Aufgabe 3.2 – Public Repositories eines Users
Rufe die öffentlichen Repositories von **`torvalds`** (Linus Torvalds, Linux-Erfinder) ab.
- Endpunkt: `GET /users/{username}/repos`
- Zeige Name, Beschreibung, Sprache und Sterne der Top-5 nach Sternanzahl

In [ ]:
# Deine Lösung hier:



In [ ]:
# MUSTERLÖSUNG 3.2
username = "torvalds"

params_repos = {"per_page": 30, "sort": "updated"}
resp_repos = requests.get(f"{GITHUB_BASE}/users/{username}/repos",
                           headers=headers, params=params_repos)
resp_repos.raise_for_status()
repos = resp_repos.json()

df_repos = pd.DataFrame([{
    'Name':        r['name'],
    'Beschreibung': (r['description'] or '')[:60],
    'Sprache':     r['language'],
    'Sterne':      r['stargazers_count'],
    'Forks':       r['forks_count']
} for r in repos])

print(f"Öffentliche Repos von @{username}: {len(df_repos)}")
print("\nTop 5 nach Sternanzahl:")
print(df_repos.sort_values('Sterne', ascending=False).head().to_string(index=False))

### Aufgabe 3.3 – Repository-Details
Rufe Details zum Repository **`torvalds/linux`** ab.
- Endpunkt: `GET /repos/{owner}/{repo}`
- Zeige: Beschreibung, Sterne, Forks, offene Issues, Standard-Branch, Erstellungsdatum

In [ ]:
# Deine Lösung hier:



In [ ]:
# MUSTERLÖSUNG 3.3
resp_linux = requests.get(f"{GITHUB_BASE}/repos/torvalds/linux", headers=headers)
resp_linux.raise_for_status()
linux = resp_linux.json()

erstellt = pd.to_datetime(linux['created_at']).strftime('%d.%m.%Y')
aktualisiert = pd.to_datetime(linux['updated_at']).strftime('%d.%m.%Y')

print(f"=== {linux['full_name']} ===")
print(f"Beschreibung:   {linux['description']}")
print(f"Sprache:        {linux['language']}")
print(f"Sterne:         {linux['stargazers_count']:,}")
print(f"Forks:          {linux['forks_count']:,}")
print(f"Offene Issues:  {linux['open_issues_count']:,}")
print(f"Standard-Branch: {linux['default_branch']}")
print(f"Erstellt am:    {erstellt}")
print(f"Zuletzt aktiv:  {aktualisiert}")

### Aufgabe 3.4 – Mehrere User vergleichen
Schreibe eine Funktion `get_user_stats(username)`, die folgende Daten eines GitHub-Users zurückgibt:
- Endpunkt: `GET /users/{username}`
- Felder: `login`, `name`, `public_repos`, `followers`, `following`

Rufe die Funktion für folgende Users auf und erstelle einen Vergleichs-DataFrame:
`["torvalds", "gvanrossum", "antirez"]`

⚠️ **Achtung Rate Limit:** Füge zwischen den Requests ein `time.sleep(1)` ein.

In [ ]:
# Ihre Lösung hier:



In [ ]:
# MUSTERLÖSUNG 3.4
def get_user_stats(username):
    """Ruft GitHub-User-Statistiken ab und gibt ein dict zurück."""
    resp = requests.get(f"{GITHUB_BASE}/users/{username}", headers=headers)
    resp.raise_for_status()
    u = resp.json()
    return {
        'Username':   u['login'],
        'Name':       u.get('name', 'N/A'),
        'Public Repos': u['public_repos'],
        'Followers':  u['followers'],
        'Following':  u['following'],
    }

users = ["torvalds", "gvanrossum", "antirez"]
stats_liste = []

for user in users:
    try:
        stats = get_user_stats(user)
        stats_liste.append(stats)
        print(f"✓ {user} geladen")
    except requests.HTTPError as e:
        print(f"✗ Fehler bei {user}: {e}")
    time.sleep(1)  # Rate Limit schonen

df_users = pd.DataFrame(stats_liste)
print("\nVergleich:")
print(df_users.to_string(index=False))